<a href="https://colab.research.google.com/github/chijike3/Bear-Image-classifier/blob/main/Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch transformers peft torchvision


In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel # to download the transformer model and read its configuration file
from peft import LoraConfig, get_peft_model
from transformers import ViTModel
from torchvision.models import resnet50
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModel, AutoTokenizer
from torchvision import models
import timm
from fastai.vision.all import *
from fastai.text.all import *
import pandas as pd
import numpy as np



In [ ]:

#--------------------LOAD SLAKE DATASET-------------------------------


import pandas as pd

splits = {'train': 'train.json', 'validation': 'validation.json', 'test': 'test.json'}
df_train = pd.read_json("hf://datasets/BoKelvin/SLAKE/" + splits["train"])
df_val = pd.read_json("hf://datasets/BoKelvin/SLAKE/" + splits["validation"])
df_combined = pd.concat([df_train, df_val])
val_idx = list(range(len(df_train), len(df_combined)))




#---MODEL SKELETAL DEFINITON---

class Multimodalmodel(nn.Module):
  def __init__(self, vision_model, text_model, num_of_answers, projection_dim=512, ):
    super(Multimodalmodel, self).__init__()
    self.vision_model = vision_model
    self.text_model = text_model
    # automatically dectect the imput dimension and change it to 512
    self.vision_proj = nn.LazyLinear(projection_dim)
    self.text_proj = nn.LazyLinear(projection_dim)
#define the classifier
    self.classifier = nn.Sequential(
        nn.Linear(projection_dim*2, 512),
        nn.ReLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(0.3),
        nn.Linear(512, num_of_answers)
    )

  def forward(self,image, text_data):
    text_idx, attention_mask = text_data
#process the image features
    image_features = self.vision_model(image)
    image_features = image_features.view(image_features.size(0), -1) # this flattens the image features to a 2d vector
    image_features = self.vision_proj(image_features)
#process the text features
    text_features = self.text_model(input_ids=text_idx, attention_mask=attention_mask)
    text_features = text_features.last_hidden_state
    text_features = text_features[:, 0, :] # here we take the CLS token
    text_features = self.text_proj(text_features)

    combined_features = torch.cat([image_features, text_features], dim=1)
    return self.classifier(combined_features)


#---- DEFINING THE DIFFERENT VARIANTS USING THE SKELETAL MODEL-----

# --- Variant 1 (BioBERT + ResNet50)------
biobert_model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
#cofigure Lora for biobert_model
bio_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)
#Apply lora cofiguration on biobert
biobert_model = get_peft_model(biobert_model, bio_config)

#configure Lora for restnet34
# strip the head of the restnet34 and take the list of the remaining layer and put them back into a sequential container
resnet_model = nn.Sequential(*list(models.resnet34(weights='DEFAULT').children())[:-1])
# cofigure Lora for the restnet34
vision_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["conv1"], # Target the convolutional layers
    lora_dropout=0.1,
    bias="none"
)

restnet_model = get_peft_model(resnet_model, vision_lora_config)

# plug into multimodalmodel skeleton
model_variant_1 = Multimodalmodel(
    vision_model=resnet_model,
    text_model=biobert_model,
    num_of_answers=   228
)

    #--------------------VARIANT 2 --------------------------------


# 1. Load PubMedBERT and Tokenizer
model_id = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
pubmed_tokz = AutoTokenizer.from_pretrained(model_id)
pubmed_base = AutoModel.from_pretrained(model_id)

# 2. Configure LoRA for pubmedbert
pubmed_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

# 3. Wrap it
lora_pubmed = get_peft_model(pubmed_base, pubmed_lora_config)




# 1. Load ViT-Base (stripping the head with num_classes=0)
vit_base = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)

# 2. Configure LoRA for Vision Transformer
# In ViT, these are usually named 'qkv' or 'query'/'value' depending on the library
vit_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv"], # In timm ViT, Query/Key/Value are often bundled as 'qkv'
    lora_dropout=0.1,
    bias="none"
)

# 3. Wrap it
lora_vit = get_peft_model(vit_base, vit_lora_config)


# Create Variant 2 using the multimodalmodel
model_variant_2 = Multimodalmodel(
    vision_model=lora_vit,
    text_model=lora_pubmed,
    num_of_answers=228
)


#---------------------VARIANT 3---------------------


# 1. Load BioBERT
bio_tokz = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
bio_base = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

# 2. BioBERT LoRA Config
bio_lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1, bias="none"
)
lora_bio_v3 = get_peft_model(bio_base, bio_lora_config)

# 1. Load ViT-Base (No Head)
vit_base_v3 = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)

# 2. ViT LoRA Config
vit_lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["qkv"],
    lora_dropout=0.1, bias="none"
)

# 3. Wrap it
lora_vit_v3 = get_peft_model(vit_base_v3, vit_lora_config)
#create the 3rd variant using the
model_variant_3 = Multimodalmodel(
    vision_model=lora_vit_v3,
    text_model=lora_bio_v3,
    num_of_answers =228
)


# initialise the lazy linear layer
my_variants = [model_variant_1, model_variant_2, model_variant_3]

# 2. Run the shared loop
for i, m in enumerate(my_variants):
    dummy_img = torch.randn(2, 3, 224, 224)
    dummy_ids = torch.randint(0, 100, (2, 64))
    dummy_mask = torch.ones(2, 64)

    # Set to eval mode just for the dummy pass to be extra safe
    m.eval()
    with torch.no_grad():
        _ = m(dummy_img, dummy_ids, dummy_mask)
    m.train() # Switch back to training mode

    print(f"Variant {i+1} (model_v{i+1}) is now fully built and ready!")



#--------------------USINg FASTAI API TO DEFINE A DATALOADER---------------

#Define the tokenization transform for the the biobert and pubmedbert text encoders
class TokenizeTransform(Transform):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def encodes(self, text):
        #  return only the input_ids and attention_mask as tensors
        tok = self.tokenizer(text, padding='max_length', truncation=True, max_length=64, return_tensors="pt")
        return tok['input_ids'].squeeze(0), tok['attention_mask'].squeeze(0)

# Create the specific transforms for each tokenizer
tfm_bio = TokenizeTransform(bio_tokz)
tfm_pubmed = TokenizeTransform(pubmed_tokz)

# define a fastai Datablock to cater for biobert text model and pubmedbert model

bio_db = DataBlock(
    blocks=(ImageBlock, TransformBlock(type_tfms=tfm_bio), CategoryBlock),
    get_x = [ColReader('img_name'), ColReader('question')], # defines the image and text input
    get_y = ColReader('answer'), # defines the label
    splitter=IndexSplitter(val_idx),
    item_tfms=Resize(224),
    n_inp = 2 ,# this tells fastai to use the two items from get_x as input to the forward method
    batch_tfms=aug_transforms(mult=1.0, do_flip=True, flip_vert=True, max_rotate=10.0) # Augmentation Strategy
)

dls_bio = bio_db.dataloaders(df_combined, bs=32)


pubmed_block = DataBlock(
    blocks=(ImageBlock, TransformBlock(type_tfms=tfm_pubmed), CategoryBlock),
    get_x=[ColReader('image_path'), ColReader('question')],
    get_y=ColReader('answer'),
    splitter= IndexSplitter(val_idx), # tells fastai the validation columnn
    item_tfms=[Resize(224)],
    batch_tfms=[Normalize.from_stats(*imagenet_stats)] # Normalization Strategy
)

dls_pubmed = pubmed_block.dataloaders(df_combined, bs=32)



